# Chapter 4 (extra) — The block (extended) Thomas algorithm

The scalar Thomas algorithm of Chapter 4 solves a tridiagonal system
$A\,u = b$ in $O(n)$ work by an LU sweep without pivoting. When several
species are coupled at every grid node — for example $O$, $R$ and an
intermediate in a homogeneous chemical reaction, or the real and
imaginary parts of an AC problem — the unknown at each node is itself a
*vector* and the matrix becomes **block-tridiagonal**: each scalar entry
$x_i, y_i, z_i$ is replaced by a $p\times p$ block $\mathbf{X}_i,
\mathbf{Y}_i, \mathbf{Z}_i$, and each $b_i$ by a length-$p$ vector.

Honeychurch (SERM §4.3, *Extending the Thomas algorithm*) notes that
Rudolph's general simulator (the basis of DigiSim) uses exactly this
block form. The derivation is the scalar one with every division
replaced by a matrix inverse and every product by a matrix product. This
notebook re-implements it idiomatically in NumPy as a 3-D-array engine
(`X`, `Y`, `Z` of shape `(m, p, p)`), which the later coupled chapters
build on.


In [1]:
import os, sys

# Walk up from the notebook's working directory to the repo root (the directory
# that contains the ``serm`` package); works whether this notebook is run from
# notebooks/ or notebooks/extras/.
_d = os.getcwd()
while not os.path.isdir(os.path.join(_d, "serm")) and os.path.dirname(_d) != _d:
    _d = os.path.dirname(_d)
sys.path.insert(0, _d)

# %matplotlib inline embeds figures and makes plt.show() a harmless no-op
# under headless (Agg) execution.
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

import serm
from serm import echem
from serm.tridiagonal import tridiag_solve

np.set_printoptions(precision=6, suppress=True)


## 1. Block LU without pivoting

Write the block system as $\mathbf{M}\,\mathbf{u}=\mathbf{b}$ with
block sub-, main- and super-diagonals $\mathbf{X}_i,\mathbf{Y}_i,
\mathbf{Z}_i$. Factor $\mathbf{M}=\mathbf{L}\,\mathbf{U}$ where
$\mathbf{U}$ has identity blocks on its diagonal and $\mathbf{Z}_i$
above it, and $\mathbf{L}$ has the pivot blocks
$\boldsymbol{\alpha}_i$ on its diagonal and $\mathbf{X}_i$ below.
Matching blocks gives the recurrences

$$\boldsymbol{\alpha}_1=\mathbf{Y}_1,\qquad
\boldsymbol{\alpha}_i=\mathbf{Y}_i-\mathbf{X}_i\,
\boldsymbol{\alpha}_{i-1}^{-1}\,\mathbf{Z}_{i-1}.$$

**Forward substitution** ($\mathbf{L}\,\mathbf{f}=\mathbf{b}$):

$$\mathbf{f}_1=\boldsymbol{\alpha}_1^{-1}\mathbf{b}_1,\qquad
\mathbf{f}_i=\boldsymbol{\alpha}_i^{-1}
\bigl(\mathbf{b}_i-\mathbf{X}_i\,\mathbf{f}_{i-1}\bigr).$$

**Back substitution** ($\mathbf{U}\,\mathbf{u}=\mathbf{f}$):

$$\mathbf{u}_m=\mathbf{f}_m,\qquad
\mathbf{u}_i=\mathbf{f}_i-\boldsymbol{\alpha}_i^{-1}\,
\mathbf{Z}_i\,\mathbf{u}_{i+1}.$$

For $p=1$ every block is a scalar and these collapse to the ordinary
Thomas recurrences — the property we exploit for validation. Rather than
forming each $\boldsymbol{\alpha}_i^{-1}$ explicitly we solve the small
$p\times p$ systems with `numpy.linalg.solve`, which is both faster and
better conditioned than an explicit inverse.


In [2]:
def block_tridiag_solve(
    X: np.ndarray, Y: np.ndarray, Z: np.ndarray, b: np.ndarray
) -> np.ndarray:
    """Solve a block-tridiagonal system ``M u = b`` (extended Thomas).

    Parameters
    ----------
    X : ndarray, shape (m-1, p, p)
        Block sub-diagonal: ``X[i]`` is the block ``M[i+1, i]``.
    Y : ndarray, shape (m, p, p)
        Block main diagonal.
    Z : ndarray, shape (m-1, p, p)
        Block super-diagonal: ``Z[i]`` is the block ``M[i, i+1]``.
    b : ndarray, shape (m, p)
        Right-hand side, one length-``p`` vector per block row.

    Returns
    -------
    ndarray, shape (m, p)
        Solution blocks ``u``.

    Notes
    -----
    No pivoting (matching the scalar Thomas algorithm); intended for the
    diagonally dominant blocks that arise from implicit finite-difference
    diffusion-reaction problems. For ``p == 1`` this reduces exactly to
    the scalar Thomas algorithm.
    """
    Y = np.asarray(Y, dtype=float)
    b = np.asarray(b, dtype=float)
    m, p, _ = Y.shape
    if X.shape[0] != m - 1 or Z.shape[0] != m - 1:
        raise ValueError("X and Z must have m-1 blocks")
    if b.shape != (m, p):
        raise ValueError("b must have shape (m, p)")

    alpha = np.empty((m, p, p))
    f = np.empty((m, p))
    alpha[0] = Y[0]
    f[0] = np.linalg.solve(alpha[0], b[0])
    for i in range(1, m):
        # alpha_i = Y_i - X_i . alpha_{i-1}^{-1} . Z_{i-1}
        w = np.linalg.solve(alpha[i - 1], Z[i - 1])   # alpha_{i-1}^{-1} Z_{i-1}
        alpha[i] = Y[i] - X[i - 1] @ w
        f[i] = np.linalg.solve(alpha[i], b[i] - X[i - 1] @ f[i - 1])

    u = np.empty((m, p))
    u[-1] = f[-1]
    for i in range(m - 2, -1, -1):
        # u_i = f_i - alpha_i^{-1} . Z_i . u_{i+1}
        u[i] = f[i] - np.linalg.solve(alpha[i], Z[i] @ u[i + 1])
    return u


## 2. A worked $2\times2$-block example

A coupled pair $O\rightleftharpoons R$ diffusing with a first-order
homogeneous interconversion gives, after an implicit (backward-Euler)
discretisation, exactly such a block system. Here we build a small
deterministic block-tridiagonal matrix with $p=2$, $m=6$ and check the
engine against an explicit dense solve assembled from the same blocks.


In [3]:
rng = np.random.default_rng(4)
m, p = 6, 2

# Build diagonally dominant random blocks so the no-pivot solve is stable.
def diag_dominant_block(p):
    A = rng.standard_normal((p, p))
    A += p * np.eye(p)                      # push weight onto the diagonal
    return A

Y = np.stack([3.0 * diag_dominant_block(p) for _ in range(m)])
X = np.stack([0.4 * rng.standard_normal((p, p)) for _ in range(m - 1)])
Z = np.stack([0.4 * rng.standard_normal((p, p)) for _ in range(m - 1)])
b = rng.standard_normal((m, p))

u_block = block_tridiag_solve(X, Y, Z, b)

# Assemble the equivalent dense (m*p) x (m*p) matrix for a reference solve.
M = np.zeros((m * p, m * p))
for i in range(m):
    M[i*p:(i+1)*p, i*p:(i+1)*p] = Y[i]
for i in range(m - 1):
    M[(i+1)*p:(i+2)*p, i*p:(i+1)*p] = X[i]        # sub-diagonal block
    M[i*p:(i+1)*p, (i+1)*p:(i+2)*p] = Z[i]        # super-diagonal block
u_dense = np.linalg.solve(M, b.reshape(-1)).reshape(m, p)

err = np.max(np.abs(u_block - u_dense))
print(f"max |block-Thomas - dense solve| = {err:.2e}")


max |block-Thomas - dense solve| = 1.67e-16


## 3. Reduction to the scalar Thomas algorithm ($p=1$)

The strongest internal check is the reduction-to-validated-limit: with
$1\times1$ blocks the engine must reproduce, bit-for-bit up to rounding,
the already-validated scalar `serm.tridiagonal.tridiag_solve`. We use the
same constant-diagonal test system as the scalar module's own port
(sub/super $=-1$, main $=2$).


In [4]:
n = 50
x = -np.ones(n - 1)          # scalar sub-diagonal
y = 2.0 * np.ones(n)         # scalar main diagonal
z = -np.ones(n - 1)          # scalar super-diagonal
b_scalar = np.ones(n)
b_scalar[-1] = 1.5

u_scalar = tridiag_solve(x, y, z, b_scalar)

# Same system promoted to 1x1 blocks.
X1 = x.reshape(n - 1, 1, 1)
Y1 = y.reshape(n, 1, 1)
Z1 = z.reshape(n - 1, 1, 1)
b1 = b_scalar.reshape(n, 1)
u_1block = block_tridiag_solve(X1, Y1, Z1, b1).ravel()

reduction_err = np.max(np.abs(u_1block - u_scalar))
print(f"max |block(p=1) - scalar Thomas| = {reduction_err:.2e}")


max |block(p=1) - scalar Thomas| = 0.00e+00


## 4. Validation

Two assert-backed checks, strongest first per the project validation
policy:

1. **Reduction to a validated limit (tier 2).** For $p=1$ the block
   engine reproduces `serm.tridiagonal.tridiag_solve` (itself a port of
   the SERM scalar solver) to machine precision.
2. **Two-implementation cross-check (tier 3).** For a $p=2$ block system
   the engine agrees with an independent dense `numpy.linalg.solve` of
   the assembled matrix to machine precision.


In [5]:
# --- Validation 1: reduction to the scalar Thomas algorithm (tier 2) ---
assert reduction_err < 1e-10, "block(p=1) disagrees with scalar Thomas"
print(f"PASS: block engine with p=1 reproduces scalar Thomas "
      f"(max err {reduction_err:.2e}).")

# --- Validation 2: block solve vs. dense reference (tier 3) ---
assert err < 1e-9, "block solve disagrees with dense reference"
print(f"PASS: 2x2-block solve matches dense numpy.linalg.solve "
      f"(max err {err:.2e}).")


PASS: block engine with p=1 reproduces scalar Thomas (max err 0.00e+00).
PASS: 2x2-block solve matches dense numpy.linalg.solve (max err 1.67e-16).


## 5. Summary

The block (extended) Thomas algorithm generalises the scalar LU sweep to
block-tridiagonal systems by replacing scalar division with a small
$p\times p$ solve. The NumPy implementation above (`block_tridiag_solve`)
is the engine that the coupled-reaction (Chapter 13) and multi-species
problems require: each grid node carries a length-$p$ concentration
vector and the implicit step is a single block solve. It was validated by
reduction to the already-validated scalar solver ($p=1$) and by an
independent dense cross-check ($p=2$).


<!-- nav-footer -->

---

[← Chapter 4 — Other Numerical Methods](../04_other_numerical_methods.ipynb)

[Contents (README)](../../README.md)